In [1]:
!pip install Sastrawi
import re
import math
import pandas as pd
import numpy as np

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

stopword = StopWordRemoverFactory().get_stop_words()


def preprocessing(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()

    tokens = [
        token for token in tokens
        if token not in stopword
    ]

    return tokens

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 1.0 MB/s eta 0:00:00


In [2]:
dokumen = [
    "Sistem komputer digunakan untuk mengolah data dan menjalankan berbagai aplikasi.",
    "Jaringan komputer menghubungkan perangkat untuk bertukar data dan informasi.",
    "Kecerdasan buatan membantu komputer menganalisis data dan memberikan solusi.",
    "Sistem temu kembali informasi digunakan untuk mencari dokumen berdasarkan kata kunci."
]

data_token = []

for i, dok in enumerate(dokumen, start=1):
    token = preprocessing(dok)
    data_token.append(token)

    print(f"Dokumen {i}")
    print("Sebelum :", dok)
    print("Sesudah :", token)
    print()

Dokumen 1
Sebelum : Sistem komputer digunakan untuk mengolah data dan menjalankan berbagai aplikasi.
Sesudah : ['sistem', 'komputer', 'digunakan', 'mengolah', 'data', 'menjalankan', 'berbagai', 'aplikasi']

Dokumen 2
Sebelum : Jaringan komputer menghubungkan perangkat untuk bertukar data dan informasi.
Sesudah : ['jaringan', 'komputer', 'menghubungkan', 'perangkat', 'bertukar', 'data', 'informasi']

Dokumen 3
Sebelum : Kecerdasan buatan membantu komputer menganalisis data dan memberikan solusi.
Sesudah : ['kecerdasan', 'buatan', 'membantu', 'komputer', 'menganalisis', 'data', 'memberikan', 'solusi']

Dokumen 4
Sebelum : Sistem temu kembali informasi digunakan untuk mencari dokumen berdasarkan kata kunci.
Sesudah : ['sistem', 'temu', 'informasi', 'digunakan', 'mencari', 'dokumen', 'berdasarkan', 'kata', 'kunci']



In [3]:
kosakata = sorted(set(
    kata
    for token_dokumen in data_token
    for kata in token_dokumen
))

bow = []

for token_dokumen in data_token:
    baris = []

    for kata in kosakata:
        baris.append(token_dokumen.count(kata))

    bow.append(baris)

bow_df = pd.DataFrame(
    bow,
    index=["D1", "D2", "D3", "D4"],
    columns=kosakata
)

print("BAG-OF-WORDS")
display(bow_df)

BAG-OF-WORDS


,aplikasi,berbagai,berdasarkan,bertukar,buatan,data,digunakan,dokumen,informasi,jaringan,...,memberikan,mencari,menganalisis,menghubungkan,mengolah,menjalankan,perangkat,sistem,solusi,temu
D1,1,1,0,0,0,1,1,0,0,0,...,0,0,0,0,1,1,0,1,0,0
D2,0,0,0,1,0,1,0,0,1,1,...,0,0,0,1,0,0,1,0,0,0
D3,0,0,0,0,1,1,0,0,0,0,...,1,0,1,0,0,0,0,0,1,0
D4,0,0,1,0,0,0,1,1,1,0,...,0,1,0,0,0,0,0,1,0,1


In [4]:
tf = []

for token_dokumen in data_token:

    jumlah_token = len(token_dokumen)
    baris_tf = []

    for kata in kosakata:

        frekuensi = token_dokumen.count(kata)

        if jumlah_token > 0:
            nilai_tf = frekuensi / jumlah_token
        else:
            nilai_tf = 0

        baris_tf.append(nilai_tf)

    tf.append(baris_tf)

tf_df = pd.DataFrame(
    tf,
    index=["D1", "D2", "D3", "D4"],
    columns=kosakata
)

print("TERM FREQUENCY (TF)")
display(tf_df.round(4))

TERM FREQUENCY (TF)


,aplikasi,berbagai,berdasarkan,bertukar,buatan,data,digunakan,dokumen,informasi,jaringan,...,memberikan,mencari,menganalisis,menghubungkan,mengolah,menjalankan,perangkat,sistem,solusi,temu
D1,0.125,0.125,0.0000,0.0000,0.000,0.1250,0.1250,0.0000,0.0000,0.0000,...,0.000,0.0000,0.000,0.0000,0.125,0.125,0.0000,0.1250,0.000,0.0000
D2,0.000,0.000,0.0000,0.1429,0.000,0.1429,0.0000,0.0000,0.1429,0.1429,...,0.000,0.0000,0.000,0.1429,0.000,0.000,0.1429,0.0000,0.000,0.0000
D3,0.000,0.000,0.0000,0.0000,0.125,0.1250,0.0000,0.0000,0.0000,0.0000,...,0.125,0.0000,0.125,0.0000,0.000,0.000,0.0000,0.0000,0.125,0.0000
D4,0.000,0.000,0.1111,0.0000,0.000,0.0000,0.1111,0.1111,0.1111,0.0000,...,0.000,0.1111,0.000,0.0000,0.000,0.000,0.0000,0.1111,0.000,0.1111


In [5]:
df = []

for kata in kosakata:

    jumlah_muncul = sum(
        kata in token_dokumen
        for token_dokumen in data_token
    )

    df.append(jumlah_muncul)

df_df = pd.DataFrame({
    "Term": kosakata,
    "DF": df
})

print("DOCUMENT FREQUENCY (DF)")
display(df_df)

DOCUMENT FREQUENCY (DF)


,Term,DF
0,aplikasi,1
1,berbagai,1
2,berdasarkan,1
3,bertukar,1
4,buatan,1
5,data,3
6,digunakan,2
7,dokumen,1
8,informasi,2
9,jaringan,1


In [6]:
jumlah_dokumen = len(data_token)

idf = []

for nilai_df in df:

    nilai_idf = math.log(
        jumlah_dokumen / nilai_df
    )

    idf.append(nilai_idf)

idf_df = pd.DataFrame({
    "Term": kosakata,
    "DF": df,
    "IDF": idf
})

print("INVERSE DOCUMENT FREQUENCY (IDF)")
display(idf_df.round(4))

INVERSE DOCUMENT FREQUENCY (IDF)


,Term,DF,IDF
0,aplikasi,1,1.3863
1,berbagai,1,1.3863
2,berdasarkan,1,1.3863
3,bertukar,1,1.3863
4,buatan,1,1.3863
5,data,3,0.2877
6,digunakan,2,0.6931
7,dokumen,1,1.3863
8,informasi,2,0.6931
9,jaringan,1,1.3863


In [7]:
tfidf_manual = []

for baris_tf in tf:

    baris_tfidf = []

    for nilai_tf, nilai_idf in zip(baris_tf, idf):

        nilai = nilai_tf * nilai_idf
        baris_tfidf.append(nilai)

    tfidf_manual.append(baris_tfidf)

tfidf_manual_df = pd.DataFrame(
    tfidf_manual,
    index=["D1", "D2", "D3", "D4"],
    columns=kosakata
)

print("TF-IDF MANUAL")
display(tfidf_manual_df.round(4))

TF-IDF MANUAL


,aplikasi,berbagai,berdasarkan,bertukar,buatan,data,digunakan,dokumen,informasi,jaringan,...,memberikan,mencari,menganalisis,menghubungkan,mengolah,menjalankan,perangkat,sistem,solusi,temu
D1,0.1733,0.1733,0.000,0.000,0.0000,0.0360,0.0866,0.000,0.000,0.000,...,0.0000,0.000,0.0000,0.000,0.1733,0.1733,0.000,0.0866,0.0000,0.000
D2,0.0000,0.0000,0.000,0.198,0.0000,0.0411,0.0000,0.000,0.099,0.198,...,0.0000,0.000,0.0000,0.198,0.0000,0.0000,0.198,0.0000,0.0000,0.000
D3,0.0000,0.0000,0.000,0.000,0.1733,0.0360,0.0000,0.000,0.000,0.000,...,0.1733,0.000,0.1733,0.000,0.0000,0.0000,0.000,0.0000,0.1733,0.000
D4,0.0000,0.0000,0.154,0.000,0.0000,0.0000,0.0770,0.154,0.077,0.000,...,0.0000,0.154,0.0000,0.000,0.0000,0.0000,0.000,0.0770,0.0000,0.154


In [8]:
dokumen_preprocessed = [
    ' '.join(token)
    for token in data_token
]

vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    norm=None,
    smooth_idf=False
)

matriks_tfidf = vectorizer.fit_transform(
    dokumen_preprocessed
)

tfidf_sklearn_df = pd.DataFrame(
    matriks_tfidf.toarray(),
    index=["D1", "D2", "D3", "D4"],
    columns=vectorizer.get_feature_names_out()
)

print("TF-IDF SCIKIT-LEARN")
display(tfidf_sklearn_df.round(4))

TF-IDF SCIKIT-LEARN


,aplikasi,berbagai,berdasarkan,bertukar,buatan,data,digunakan,dokumen,informasi,jaringan,...,memberikan,mencari,menganalisis,menghubungkan,mengolah,menjalankan,perangkat,sistem,solusi,temu
D1,2.3863,2.3863,0.0000,0.0000,0.0000,1.2877,1.6931,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,2.3863,2.3863,0.0000,1.6931,0.0000,0.0000
D2,0.0000,0.0000,0.0000,2.3863,0.0000,1.2877,0.0000,0.0000,1.6931,2.3863,...,0.0000,0.0000,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.0000,0.0000,2.3863,1.2877,0.0000,0.0000,0.0000,0.0000,...,2.3863,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000
D4,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,1.6931,2.3863,1.6931,0.0000,...,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,1.6931,0.0000,2.3863


In [9]:
tfidf_sklearn_df = tfidf_sklearn_df[
    tfidf_manual_df.columns
]

perbandingan = pd.DataFrame({
    "TF-IDF Manual": tfidf_manual_df.values.flatten(),
    "TF-IDF Scikit-Learn": tfidf_sklearn_df.values.flatten()
})

print("PERBANDINGAN TF-IDF")
display(perbandingan.round(4))

PERBANDINGAN TF-IDF


,TF-IDF Manual,TF-IDF Scikit-Learn
0,0.1733,2.3863
1,0.1733,2.3863
2,0.0000,0.0000
3,0.0000,0.0000
4,0.0000,0.0000
...,...,...
95,0.0000,0.0000
96,0.0000,0.0000
97,0.0770,1.6931
98,0.0000,0.0000


In [10]:
hasil_sama = np.allclose(
    tfidf_manual_df.values,
    tfidf_sklearn_df.values
)

print("Hasil manual dan Scikit-Learn sama:", hasil_sama)

Hasil manual dan Scikit-Learn sama: False


In [11]:
print("MATRIKS TF-IDF MANUAL")
display(tfidf_manual_df.round(4))

MATRIKS TF-IDF MANUAL


,aplikasi,berbagai,berdasarkan,bertukar,buatan,data,digunakan,dokumen,informasi,jaringan,...,memberikan,mencari,menganalisis,menghubungkan,mengolah,menjalankan,perangkat,sistem,solusi,temu
D1,0.1733,0.1733,0.000,0.000,0.0000,0.0360,0.0866,0.000,0.000,0.000,...,0.0000,0.000,0.0000,0.000,0.1733,0.1733,0.000,0.0866,0.0000,0.000
D2,0.0000,0.0000,0.000,0.198,0.0000,0.0411,0.0000,0.000,0.099,0.198,...,0.0000,0.000,0.0000,0.198,0.0000,0.0000,0.198,0.0000,0.0000,0.000
D3,0.0000,0.0000,0.000,0.000,0.1733,0.0360,0.0000,0.000,0.000,0.000,...,0.1733,0.000,0.1733,0.000,0.0000,0.0000,0.000,0.0000,0.1733,0.000
D4,0.0000,0.0000,0.154,0.000,0.0000,0.0000,0.0770,0.154,0.077,0.000,...,0.0000,0.154,0.0000,0.000,0.0000,0.0000,0.000,0.0770,0.0000,0.154


In [12]:
for i, baris in tfidf_manual_df.iterrows():

    nilai_tertinggi = baris.max()
    term_tertinggi = baris.idxmax()

    print(
        f"{i}: term dengan bobot TF-IDF tertinggi adalah "
        f"'{term_tertinggi}' dengan nilai {nilai_tertinggi:.4f}"
    )

D1: term dengan bobot TF-IDF tertinggi adalah 'aplikasi' dengan nilai 0.1733
D2: term dengan bobot TF-IDF tertinggi adalah 'bertukar' dengan nilai 0.1980
D3: term dengan bobot TF-IDF tertinggi adalah 'buatan' dengan nilai 0.1733
D4: term dengan bobot TF-IDF tertinggi adalah 'berdasarkan' dengan nilai 0.1540


**Analisis:**

Pada D1, term 'aplikasi' memiliki bobot TF-IDF tertinggi sebesar 0.1733 karena cukup mewakili isi dokumen tentang penggunaan sistem komputer dan aplikasi. Pada D2, term 'bertukar' memiliki bobot tertinggi sebesar 0.1980 karena berkaitan dengan pembahasan jaringan komputer dan pertukaran data. Pada D3, term 'buatan' memiliki bobot tertinggi sebesar 0.1733 karena berkaitan dengan topik kecerdasan buatan. Pada D4, term 'berdasarkan' memiliki bobot tertinggi sebesar 0.1540 karena digunakan dalam konteks pencarian dokumen berdasarkan kata kunci. Term-term tersebut penting karena memiliki bobot yang tinggi dan dapat menunjukkan kata yang paling menonjol pada masing-masing dokumen.
